# Configuration 2 -> 
## Openai-embedding-small


In [57]:
import pandas as pd

In [58]:
df = pd.read_csv("/Users/midhunln/Documents/rag20march_with_eval/notebooks/datasets/openai_small/all_chunks.csv")

In [59]:
df.head()

,id,text
0,/Users/midhunln/Documents/rag20march_with_eval...,Page 1 of 132 \n \n \nMASTER CIRCULAR \n \nSEB...
1,/Users/midhunln/Documents/rag20march_with_eval...,Securities and Exchange Board of India (Issue ...
2,/Users/midhunln/Documents/rag20march_with_eval...,Page 2 of 132 \n \n4. With the issuance of thi...
3,/Users/midhunln/Documents/rag20march_with_eval...,"suffered thereunder, any right, privilege, obl..."
4,/Users/midhunln/Documents/rag20march_with_eval...,regulations and bidding portal. \n7. All liste...


In [60]:
df["id"][0]

'/Users/midhunln/Documents/rag20march_with_eval/Ingestion_plus_Retriever_eval/Documents/Master Circular for Issue of Capital and Disclosure Requirements.pdf_0_0'

In [61]:
df.info

<bound method DataFrame.info of                                                      id  \
0     /Users/midhunln/Documents/rag20march_with_eval...   
1     /Users/midhunln/Documents/rag20march_with_eval...   
2     /Users/midhunln/Documents/rag20march_with_eval...   
3     /Users/midhunln/Documents/rag20march_with_eval...   
4     /Users/midhunln/Documents/rag20march_with_eval...   
...                                                 ...   
1103  /Users/midhunln/Documents/rag20march_with_eval...   
1104  /Users/midhunln/Documents/rag20march_with_eval...   
1105  /Users/midhunln/Documents/rag20march_with_eval...   
1106  /Users/midhunln/Documents/rag20march_with_eval...   
1107  /Users/midhunln/Documents/rag20march_with_eval...   

                                                   text  
0     Page 1 of 132 \n \n \nMASTER CIRCULAR \n \nSEB...  
1     Securities and Exchange Board of India (Issue ...  
2     Page 2 of 132 \n \n4. With the issuance of thi...  
3     suffered thereunder, 

In [62]:
df_sampled = df.sample(frac = 0.1)

In [63]:
df_sampled.head()

,id,text
886,/Users/midhunln/Documents/rag20march_with_eval...,Page 201 of 261 \n \nthat could adversely affe...
535,/Users/midhunln/Documents/rag20march_with_eval...,20(3) \nDisclosure of the Secretarial Audit R...
214,/Users/midhunln/Documents/rag20march_with_eval...,b. Attention of investors to be drawn to instr...
588,/Users/midhunln/Documents/rag20march_with_eval...,comprehensive income / loss of its associates ...
56,/Users/midhunln/Documents/rag20march_with_eval...,with any other SEBI registered SCSBs. Such acc...


In [64]:
!pip install langchain-ollama


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [65]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

chat = ChatOllama(model = "llama3")

In [66]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

chat = ChatOllama(model="llama3")

def llm_call(text):
    prompt = f"""
You are a query generator. Based on the text below, generate a concise search query
that a user would likely use to find this information. The output must only have the query and nothing else.
It is very important that the output is only the query and nothing else.

Text:
{text}
"""
    response = chat.invoke([HumanMessage(content=prompt)])
    return response.content.strip()


# Apply row-wise correctly
df_sampled["query"] = df_sampled["text"].apply(llm_call)

In [67]:
df_sampled["query"].iloc[2]

'"issue structure for different investor categories"'

In [68]:
df_sampled.to_csv("/Users/midhunln/Documents/rag20march_with_eval/notebooks/datasets/openai_small/chunk_to_query_mapping.csv", index = False)

In [69]:
df_sampled.head()

,id,text,query
886,/Users/midhunln/Documents/rag20march_with_eval...,Page 201 of 261 \n \nthat could adversely affe...,long term ecological damage habitat sustainabi...
535,/Users/midhunln/Documents/rag20march_with_eval...,20(3) \nDisclosure of the Secretarial Audit R...,"""corporate governance report disclosure"""
214,/Users/midhunln/Documents/rag20march_with_eval...,b. Attention of investors to be drawn to instr...,"""issue structure for different investor catego..."
588,/Users/midhunln/Documents/rag20march_with_eval...,comprehensive income / loss of its associates ...,"""comprehensive income loss associates joint ve..."
56,/Users/midhunln/Documents/rag20march_with_eval...,with any other SEBI registered SCSBs. Such acc...,"""SEBI SCSB ASBA regulations"""


# Construct the qrels dataset for rankx

In [70]:
qrels = {}

for _, row in df_sampled.iterrows():
    query = row["query"]
    doc_id = row["id"]

    if query not in qrels:
        qrels[query] = {}

    qrels[query][doc_id] = 1  # relevance score

In [71]:
qrels

{'long term ecological damage habitat sustainability': {'/Users/midhunln/Documents/rag20march_with_eval/Ingestion_plus_Retriever_eval/Documents/Master circular for compliance with the provisions of the Securities and Exchange Board of India (Listing Obligations and Disclosure Requirements) Regulations, 2015 by listed entities.pdf_200_36': 1},
 '"corporate governance report disclosure"': {'/Users/midhunln/Documents/rag20march_with_eval/Ingestion_plus_Retriever_eval/Documents/Master circular for compliance with the provisions of the Securities and Exchange Board of India (Listing Obligations and Disclosure Requirements) Regulations, 2015 by listed entities.pdf_85_35': 1},
 '"issue structure for different investor categories"': {'/Users/midhunln/Documents/rag20march_with_eval/Ingestion_plus_Retriever_eval/Documents/Master Circular for Issue of Capital and Disclosure Requirements.pdf_88_14': 1},
 '"comprehensive income loss associates joint ventures quarterly financial results"': {'/Users/

# Start retriever


In [72]:
!pip install langchain-pinecone


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [73]:
import sys
sys.path.append('/Users/midhunln/Documents/rag20march_with_eval/Ingestion_plus_Retriever_eval')

from dotenv import load_dotenv
load_dotenv("/Users/midhunln/Documents/rag20march_with_eval/Ingestion_plus_Retriever_eval/ingestion.env")

from Repositories.pinecone_repository import PineconeRepository

In [74]:
from Repositories.pinecone_repository import PineconeConfig
from src.openai_embedding import OpenAIEmbedding
from src.sparse_embedding import SentenceTransformerSparseEmbedding

In [75]:
config = PineconeConfig()

dense_embedding_strategy = OpenAIEmbedding(config)

sparse_embedding_strategy = SentenceTransformerSparseEmbedding(config)


Loading weights: 100%|██████████| 204/204 [00:00<00:00, 31985.27it/s]
The tied weights mapping and config for this model specifies to tie bert.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BertForMaskedLM LOAD REPORT from: naver/splade-cocondenser-ensembledistil
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [76]:
import os

In [77]:
pinecone_repository = PineconeRepository(
            api_key=os.getenv("PINECONE_API_KEY"),
            environment=os.getenv("PINECONE_ENVIRONMENT", "us-east-1-aws"),
            dense_embedding_strategy=dense_embedding_strategy,
            sparse_embedding_strategy=sparse_embedding_strategy,
            pinecone_config=config
        )

In [78]:
df_sampled["relevant_docs"] = df_sampled["query"].apply(pinecone_repository.query_vector_store_for_rankx)

In [79]:
df_sampled.to_csv("/Users/midhunln/Documents/rag20march_with_eval/notebooks/datasets/openai_small/chunk_to_query_relevantdocs_mapping.csv", index = False)

In [80]:
df_sampled.head()

,id,text,query,relevant_docs
886,/Users/midhunln/Documents/rag20march_with_eval...,Page 201 of 261 \n \nthat could adversely affe...,long term ecological damage habitat sustainabi...,[{'id': '/Users/midhunln/Documents/rag20march_...
535,/Users/midhunln/Documents/rag20march_with_eval...,20(3) \nDisclosure of the Secretarial Audit R...,"""corporate governance report disclosure""",[{'id': '/Users/midhunln/Documents/rag20march_...
214,/Users/midhunln/Documents/rag20march_with_eval...,b. Attention of investors to be drawn to instr...,"""issue structure for different investor catego...",[{'id': '/Users/midhunln/Documents/rag20march_...
588,/Users/midhunln/Documents/rag20march_with_eval...,comprehensive income / loss of its associates ...,"""comprehensive income loss associates joint ve...",[{'id': '/Users/midhunln/Documents/rag20march_...
56,/Users/midhunln/Documents/rag20march_with_eval...,with any other SEBI registered SCSBs. Such acc...,"""SEBI SCSB ASBA regulations""",[{'id': '/Users/midhunln/Documents/rag20march_...


In [81]:
df_sampled["relevant_docs"].iloc[0]

[{'id': '/Users/midhunln/Documents/rag20march_with_eval/Ingestion_plus_Retriever_eval/Documents/Master circular for compliance with the provisions of the Securities and Exchange Board of India (Listing Obligations and Disclosure Requirements) Regulations, 2015 by listed entities.pdf_200_36',
  'score': 16.9695168},
 {'id': '/Users/midhunln/Documents/rag20march_with_eval/Ingestion_plus_Retriever_eval/Documents/Master circular for compliance with the provisions of the Securities and Exchange Board of India (Listing Obligations and Disclosure Requirements) Regulations, 2015 by listed entities.pdf_182_30',
  'score': 11.1270905},
 {'id': '/Users/midhunln/Documents/rag20march_with_eval/Ingestion_plus_Retriever_eval/Documents/Master circular for compliance with the provisions of the Securities and Exchange Board of India (Listing Obligations and Disclosure Requirements) Regulations, 2015 by listed entities.pdf_151_10',
  'score': 10.9569683},
 {'id': '/Users/midhunln/Documents/rag20march_wit

In [82]:
qrels_dict = {}

for i, row in df_sampled.iterrows():
    qid = str(i)
    true_doc = row["id"]   # single doc id string
    qrels_dict[qid] = {true_doc: 1}

In [83]:
run_dict = {}

for i, row in df_sampled.iterrows():
    qid = str(i)
    
    run_dict[qid] = {
        doc["id"]: float(doc["score"])
        for doc in row["relevant_docs"]
    }

In [84]:
from ranx import Qrels, Run, evaluate

qrels = Qrels(qrels_dict)
run = Run(run_dict)

metrics = [
    "mrr",
    "ndcg@10",
    "recall@5",
    "recall@10",
    "precision@5"
]

results = evaluate(qrels, run, metrics)
print(results)

{'mrr': np.float64(0.6157979407979408), 'ndcg@10': np.float64(0.6932942555188456), 'recall@5': np.float64(0.8378378378378378), 'recall@10': np.float64(0.9369369369369369), 'precision@5': np.float64(0.16756756756756752)}


In [85]:
final_result = pd.Series(results)


In [86]:
final_result

mrr            0.615798
ndcg@10        0.693294
recall@5       0.837838
recall@10      0.936937
precision@5    0.167568
dtype: float64

In [87]:
result = pd.Series(results)
result


mrr            0.615798
ndcg@10        0.693294
recall@5       0.837838
recall@10      0.936937
precision@5    0.167568
dtype: float64